In [1]:
#now that our data is clean we can proceed to build our recommendation system for our website
import pandas as pd    
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import difflib
from rapidfuzz import process
import json

In [2]:
# the work flow will be as follows
# 1. load the cleaned data
# 2. preprocess the data (make a combined features column)
# 3. compute the tf-idf matrix
# 4. compute the cosine similarity matrix
# 5. make a function to get movie recommendations using fuzzy matching or difflib
df = pd.read_csv(r'C:\Users\alyah\VC\first_project\recommendation system\movies_DB_cleaned.csv',keep_default_na=False )

In [3]:
# we will use json.loads to convert the json strings back to lists and dictionaries so we can use them
list_dict_cols = ['genres', 'keywords', 'cast', 'external_ids']
for col in list_dict_cols:
    df[col] = df[col].apply(json.loads)

In [4]:
list_cols=['genres','keywords',"cast"]
for col in list_cols:
    print(f"{col} : {type(df[col].iloc[0])}")
dict_cols=['external_ids']
for col in dict_cols:
    print(f"{col} : {type(df[col].iloc[0])}")

genres : <class 'list'>
keywords : <class 'list'>
cast : <class 'list'>
external_ids : <class 'dict'>


In [5]:
df.head()

,movie_id,title,overview,genres,keywords,director,release_date,runtime,popularity,poster_url,budget,revenue,tagline,external_ids,certification,cast
0,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[Family, Comedy, Animation, Adventure]","[rescue, friendship, mission, jealousy, villai...",John Lasseter,1995-11-22,81,18.5540,https://image.tmdb.org/t/p/w500/uXDfjJbdP4ijW5...,30000000,394436586,The adventure takes off when toys come to life!,"{'id': 862, 'imdb_id': 'tt0114709', 'wikidata_...",G,"[{'name': 'Tom Hanks', 'profile_url': 'https:/..."
1,8844,Jumanji,When siblings Judy and Peter discover an encha...,"[Adventure, Fantasy, Family]","[giant insect, board game, disappearance, jung...",Joe Johnston,1995-12-15,104,2.5713,https://image.tmdb.org/t/p/w500/iWV47r6kFneCiA...,65000000,262821940,It's a jungle in here.,"{'id': 8844, 'imdb_id': 'tt0113497', 'wikidata...",PG,"[{'name': 'Robin Williams', 'profile_url': 'ht..."
2,15602,Grumpier Old Men,A family wedding reignites the ancient feud be...,"[Romance, Comedy]","[fishing, sequel, old man, best friend, weddin...",Howard Deutch,1995-12-22,101,2.5156,https://image.tmdb.org/t/p/w500/1FSXpj5e8l4KH6...,25000000,71500000,Still Yelling. Still Fighting. Still Ready for...,"{'id': 15602, 'imdb_id': 'tt0113228', 'wikidat...",PG-13,"[{'name': 'Walter Matthau', 'profile_url': 'ht..."
3,31357,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...","[Comedy, Drama, Romance]","[based on novel or book, single mother, divorc...",Forest Whitaker,1995-12-22,127,3.4707,https://image.tmdb.org/t/p/w500/qJU6rfil5xLVb5...,16000000,81452156,Friends are the people who let you be yourself...,"{'id': 31357, 'imdb_id': 'tt0114885', 'wikidat...",R,"[{'name': 'Whitney Houston', 'profile_url': 'h..."
4,11862,Father of the Bride Part II,Just when George Banks has recovered from his ...,"[Comedy, Family]","[daughter, baby, parent child relationship, mi...",Charles Shyer,1995-12-08,106,2.9373,https://image.tmdb.org/t/p/w500/rj4LBtwQ0uGrpB...,0,76594107,Just when his world is back to normal... he's ...,"{'id': 11862, 'imdb_id': 'tt0113041', 'wikidat...",PG,"[{'name': 'Steve Martin', 'profile_url': 'http..."


In [6]:
df.isnull().sum()

movie_id         0
title            0
overview         0
genres           0
keywords         0
director         0
release_date     0
runtime          0
popularity       0
poster_url       0
budget           0
revenue          0
tagline          0
external_ids     0
certification    0
cast             0
dtype: int64

In [7]:
df.sort_values('popularity', ascending=False).head(100)

,movie_id,title,overview,genres,keywords,director,release_date,runtime,popularity,poster_url,budget,revenue,tagline,external_ids,certification,cast
91183,798645,The Running Man,"Desperate to save his sick daughter, working-c...","[Action, Thriller, Science Fiction]","[based on novel or book, dark comedy, survival...",Edgar Wright,2025-11-11,133,446.1148,https://image.tmdb.org/t/p/w500/dKL78O9zxczVgj...,110000000,68391082,Hunt him down.,"{'id': 798645, 'imdb_id': 'tt14107334', 'wikid...",R,"[{'name': 'Glen Powell', 'profile_url': 'https..."
91184,1084242,Zootopia 2,After cracking the biggest case in Zootopia's ...,"[Animation, Comedy, Adventure, Family, Mystery]","[snake, bunny, fox, cop, sequel, anthropomorph...",Jared Bush,2025-11-26,107,392.3259,https://image.tmdb.org/t/p/w500/oJ7g2CifqpStmo...,150000000,1137444817,Zootopia will be changed furrrever...,"{'id': 1084242, 'imdb_id': 'tt26443597', 'wiki...",PG,"[{'name': 'Ginnifer Goodwin', 'profile_url': '..."
91185,812583,Wake Up Dead Man: A Knives Out Mystery,When young priest Jud Duplenticy is sent to as...,"[Thriller, Mystery, Comedy]","[detective, investigation, sequel, murder, who...",Rian Johnson,2025-11-26,145,348.3879,https://image.tmdb.org/t/p/w500/qCOGGi8JBVEZMc...,210000000,4000000,He works in mysterious ways.,"{'id': 812583, 'imdb_id': 'tt14364480', 'wikid...",PG-13,"[{'name': 'Daniel Craig', 'profile_url': 'http..."
21026,23527,First Squad: The Moment of Truth,Set during the opening days of World War II on...,"[Fantasy, Animation, Action, Science Fiction]","[supernatural, super soldier, russian army]",Yoshiharu Ashino,2009-05-13,73,299.4968,https://image.tmdb.org/t/p/w500/hBj1aTnGf4564K...,0,0,,"{'id': 23527, 'imdb_id': 'tt1343712', 'wikidat...",N/A,"[{'name': 'Sergei Aisman', 'profile_url': None..."
91186,1387382,Hunting Season,When a reclusive survivalist and his daughter ...,"[Action, Drama, Thriller]",[],Raja Collins,2025-12-05,93,266.0149,https://image.tmdb.org/t/p/w500/cbryTyaWdqrKpQ...,0,0,It takes an awful lot to kill a person.,"{'id': 1387382, 'imdb_id': 'tt32537226', 'wiki...",N/A,"[{'name': 'Mel Gibson', 'profile_url': 'https:..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91274,1010756,The Strangers: Chapter 2,When The Strangers learn that one of their vic...,"[Horror, Thriller]","[ambulance, sequel, hospital, slasher, reboot,...",Renny Harlin,2025-09-25,98,30.8989,https://image.tmdb.org/t/p/w500/aEk9jLbiKTVssd...,8000000,21855027,Survival was just the beginning.,"{'id': 1010756, 'imdb_id': 'tt28671344', 'wiki...",R,"[{'name': 'Madelaine Petsch', 'profile_url': '..."
91251,1511789,Captain Hook: The Cursed Tides,In the aftermath of a devastating defeat by hi...,"[Adventure, Action, Horror]","[island, english channel, channel islands, hoo...",Lars Janssen,2025-07-11,90,30.7558,https://image.tmdb.org/t/p/w500/bcP7FtskwsNp1i...,0,0,Blood flows like the tide.,"{'id': 1511789, 'imdb_id': 'tt33458086', 'wiki...",N/A,"[{'name': 'Sean Cronin', 'profile_url': 'https..."
91250,950387,A Minecraft Movie,Four misfits find themselves struggling with o...,"[Family, Fantasy, Comedy, Adventure, Action]","[friendship, surrealism, exploration, portal, ...",Jared Hess,2025-03-31,101,30.4814,https://image.tmdb.org/t/p/w500/yFHHfHcUgGAxzi...,150000000,957949195,Be there and be square.,"{'id': 950387, 'imdb_id': 'tt3566834', 'wikida...",PG,"[{'name': 'Jason Momoa', 'profile_url': 'https..."
3411,106,Predator,A team of elite commandos on a secret mission ...,"[Science Fiction, Action, Adventure, Thriller]","[guerrilla warfare, central and south america,...",John McTiernan,1987-06-12,107,29.9190,https://image.tmdb.org/t/p/w500/k3mW4qfJo6SKqe...,15000000,98267558,Soon the hunt will begin.,"{'id': 106, 'imdb_id': 'tt0093773', 'wikidata_...",R,"[{'name': 'Arnold Schwarzenegger', 'profile_ur..."


In [8]:
print(df.shape)

(95445, 16)


In [9]:
# our combined features column will include the following columns
# 'title', 'overview', 'tagline', 'genres', 'keywords', 'cast', 'director' as its the standard features used in content-based movie recommendation systems
recommendation_system = pd.DataFrame()

def join_names_from_cast(cast):

    list_cast= []
    for member in cast[:5]:  # get first 5 cast members
        if 'name' in member:
            list_cast.append(member['name'])
    return " ".join(list_cast) # join names of first 5 cast members

recommendation_system['combined_features'] = (
    df['overview'] + " " +
    df['tagline'] + " " +
    df['genres'].apply(lambda x: " ".join(x)) + " " +
    df['keywords'].apply(lambda x: " ".join(x)) + " " +
    df['cast'].apply(join_names_from_cast) + " " +
    df['director']
)


In [10]:
recommendation_system['combined_features'].head(10).to_list()

["Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences. The adventure takes off when toys come to life! Family Comedy Animation Adventure rescue friendship mission jealousy villain bullying elementary school rivalry anthropomorphism friends computer animation buddy walkie talkie toy car boy next door new toy neighborhood toy comes to life resourcefulness toy pixar Tom Hanks Tim Allen Don Rickles Jim Varney Wallace Shawn John Lasseter",
 "When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's been trapped inside the game for 26 years -- into their living room. Alan's only hope for freedom is to finish the game, which proves risky as all three find t

In [11]:
recommendation_system['combined_features'].isnull().sum()  

np.int64(0)

In [12]:
recommendation_system.shape

(95445, 1)

In [13]:
recommendation_system.head()

,combined_features
0,"Led by Woody, Andy's toys live happily in his ..."
1,When siblings Judy and Peter discover an encha...
2,A family wedding reignites the ancient feud be...
3,"Cheated on, mistreated and stepped on, the wom..."
4,Just when George Banks has recovered from his ...


In [14]:
# nowthat we have our combined features column we can proceed to compute the tf-idf matrix
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(recommendation_system['combined_features'])
tfidf_matrix.shape

(95445, 187910)

In [15]:
print(tfidf_matrix)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4106331 stored elements and shape (95445, 187910)>
  Coords	Values
  (0, 95675)	0.07946366854042504
  (0, 181523)	0.3033043656298905
  (0, 7506)	0.24875761929800005
  (0, 168755)	0.22011391578515854
  (0, 98119)	0.060883311343225215
  (0, 69843)	0.1000334756135961
  (0, 141284)	0.07942070481783693
  (0, 19679)	0.08738648443683357
  (0, 24016)	0.07860099510624059
  (0, 26129)	0.35157733136381736
  (0, 97308)	0.15509513816726483
  (0, 146484)	0.08270165765488878
  (0, 3482)	0.10123251509245831
  (0, 99124)	0.08704105140956239
  (0, 129029)	0.06646038122818997
  (0, 71102)	0.06953310826328156
  (0, 129378)	0.10046360698427961
  (0, 33123)	0.09235387187267194
  (0, 148867)	0.09837782289656655
  (0, 123121)	0.07580200003691256
  (0, 47956)	0.09618341772932949
  (0, 53024)	0.08092741371906771
  (0, 95527)	0.07503488545682531
  (0, 10763)	0.10719029132298329
  (0, 43974)	0.09334199043660404
  :	:
  (95443, 120198)	0.190086782798253

In [16]:
# using cosine similarity won't be efficient for large datasets 
# as you will need to store the entire matrix in memory (n x n size where n is the number of movies)
# so for our dataset 95K movies the matrix will be 95K x 95K which is around 9 billion entries
# instead we can use sklearn's NearestNeighbors which uses efficient algorithms to find the nearest neighbors without computing the entire distance matrix

from sklearn.neighbors import NearestNeighbors

In [17]:
# we will use cosine metric to find the nearest neighbors
nn_model = NearestNeighbors(metric='cosine', algorithm='brute')
nn_model.fit(tfidf_matrix)

,n_neighbors,5
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [18]:
all_titles = df['title'].tolist()
print(all_titles[:10])  

['Toy Story', 'Jumanji', 'Grumpier Old Men', 'Waiting to Exhale', 'Father of the Bride Part II', 'Heat', 'Sabrina', 'Tom and Huck', 'Sudden Death', 'GoldenEye']


In [19]:
user_input= "avengers"  
substring_matches = [i for i, title in enumerate(all_titles) if user_input.lower() in title.lower()]

if substring_matches:
    substring_matches.sort(key=lambda x: df.iloc[x]['popularity'], reverse=True)
    print("Substring matches:")
    for i in substring_matches[:10]:
        print(i, all_titles[i])
    user_choice = int(input("Enter the index of the movie you meant from the above list: 1-10 "))
    idx = substring_matches[user_choice - 1] 
    print("You selected:", all_titles[idx]) 
else:
    
    match = process.extractOne(user_input.lower(), [title.lower() for title in all_titles])
    if match:
        idx = match[2]
        print("Fuzzy match:", all_titles[idx])
    else:
        print("No match found.")
        idx = None

Substring matches:
16953 The Avengers
24817 Avengers: Infinity War
24818 Avengers: Endgame
24809 Avengers: Age of Ultron
2049 The Avengers
86820 LEGO Marvel Avengers: Mission Demolition
44975 Scavengers
30238 Avengers Grimm
45262 Ultimate Avengers 2
22369 Crippled Avengers
You selected: The Avengers


In [20]:
if idx is not None:
    n_recommendations = 10
    distances, indices = nn_model.kneighbors(tfidf_matrix[idx], n_neighbors=n_recommendations+1)
    recommended_indices = indices.flatten()
    print("Recommendations:")
    for i in recommended_indices:
        print(i, df.iloc[i]['title'])

Recommendations:
16953 The Avengers
24809 Avengers: Age of Ultron
24818 Avengers: Endgame
24821 Captain America: Civil War
43167 Team Thor
24817 Avengers: Infinity War
43269 Marvel Studios: Assembling a Universe
24816 Captain Marvel
24819 Thor: Ragnarok
21128 Captain America: The Winter Soldier
14539 Iron Man 2


In [39]:
# now we will integrate this into a function 
def get_movie_recommendations(user_input, n_recommendations=10):
    substring_matches = [i for i, title in enumerate(all_titles) if user_input.lower() in title.lower()]

    if substring_matches:
        substring_matches.sort(key=lambda x: df.iloc[x]['popularity'], reverse=True)
        print("Results:")
        for i in substring_matches[:10]:
            print(i, all_titles[i])
        user_choice = int(input("Enter the index of the movie you meant from the above list: 1-10 "))
        idx = substring_matches[user_choice - 1] 
        print("You selected:", all_titles[idx]) 
    else:
        
        match = process.extractOne(user_input.lower(), [title.lower() for title in all_titles])
        if match:
            idx = match[2]
            print("Fuzzy match:", all_titles[idx])
        else:
            print("No match found.")
            return

    distances, indices = nn_model.kneighbors(tfidf_matrix[idx], n_neighbors=n_recommendations+1)
    recommended_indices = indices.flatten()
    print("Recommendations:")
    for i in recommended_indices:
        print(i, df.iloc[i]['title'])

user_input = input("Enter a movie title: ")
get_movie_recommendations(user_input, n_recommendations=10)

Results:
109 Taxi Driver
16609 Drive
4140 Driven
45960 Baby Driver
4721 Mulholland Drive
86561 Drive-Away Dolls
49675 A Taxi Driver
82837 Sexual Drive
45527 Bus Driver
15988 Drive Angry
You selected: Baby Driver
Recommendations:
45960 Baby Driver
49516 Wheelman
10392 The Driver
29589 Robbery
16609 Drive
6879 Quick Change
22630 Drive Hard
7567 The Getaway
8169 Taxi
70657 The Big Heist
1052 Bonnie and Clyde


In [22]:
# now we will work on search functionality
# user should be able to search by people involved (actors, directors) and genre

In [23]:
directors_list = df['director'].unique().tolist()
def search_by_director(director_name):
    # Case-insensitive match
    results = df[df['director'].str.lower().str.contains(director_name.lower())].sort_values(by='popularity', ascending=False).reset_index(drop=True)
    if not results.empty:
     return results[['title', 'release_date', 'director']]
    
# fuzzy match as a fallback if no direct matches found
    match = process.extractOne(director_name.lower(), [director.lower() for director in directors_list])
    if match:
        fuzzy_director = match[0]
        results = df[df['director'].str.lower() == fuzzy_director].sort_values(by='popularity', ascending=False).reset_index(drop=True)
        return results[['title', 'release_date', 'director']]
    else:
        return pd.DataFrame(columns=['title', 'release_date', 'director'])
     

In [24]:
actors_list = df['cast'].apply(lambda x: [member['name'] for member in x if 'name' in member]).explode().dropna().unique().tolist()

def search_by_actor(actor_name):
    def actor_in_cast(cast):
        return any('name' in member and actor_name.lower() in member['name'].lower() for member in cast)
    
    actor_matches = df[df['cast'].apply(actor_in_cast)].sort_values(by='popularity', ascending=False).reset_index(drop=True)
    if not actor_matches.empty:
        return actor_matches[['title', 'release_date', 'director']].reset_index(drop=True)
    
    else:
    # Fuzzy match as fallback
     match = process.extractOne(actor_name.lower(), [actor.lower() for actor in actors_list])
     if match:
        fuzzy_actor = match[0]
        actor_matches = df[df['cast'].apply(lambda cast: any('name' in member and member['name'].lower() == fuzzy_actor for member in cast))].sort_values(by='popularity', ascending=False).reset_index(drop=True)
        return actor_matches[['title', 'release_date', 'director']].reset_index(drop=True)
    
     else: # No match found
      return pd.DataFrame(columns=['title', 'release_date', 'director'])



In [25]:
genres_list = df['genres'].explode().dropna().unique().tolist()
def search_by_genre(genre_name):
    # Case-insensitive match
    results = df[df['genres'].apply(lambda x: any(genre_name.lower() in genre.lower() for genre in x))].sort_values(by='popularity', ascending=False).reset_index(drop=True)
    if not results.empty:
        return results[['title', 'release_date', 'director']]
    
    # Fuzzy match as fallback
    match = process.extractOne(genre_name.lower(), [genre.lower() for genre in genres_list])
    if match:
        fuzzy_genre = match[0]
        results = df[df['genres'].apply(lambda genres: any(genre.lower() == fuzzy_genre for genre in genres))].sort_values(by='popularity', ascending=False).reset_index(drop=True)
        return results[['title', 'release_date', 'director']]
    else:
    # No match found
     return pd.DataFrame(columns=['title', 'release_date', 'director'])

In [26]:
# test the search functions

print(search_by_director("Tarantino"))

                                 title release_date           director
0                         Pulp Fiction   1994-09-10  Quentin Tarantino
1                     Django Unchained   2012-12-25  Quentin Tarantino
2                 Inglourious Basterds   2009-08-02  Quentin Tarantino
3   Kill Bill: The Whole Bloody Affair   2011-03-27  Quentin Tarantino
4                    Kill Bill: Vol. 1   2003-10-10  Quentin Tarantino
5     Once Upon a Time... in Hollywood   2019-07-24  Quentin Tarantino
6                    Kill Bill: Vol. 2   2004-04-16  Quentin Tarantino
7                    The Hateful Eight   2015-12-25  Quentin Tarantino
8                       Reservoir Dogs   1992-09-02  Quentin Tarantino
9                         Jackie Brown   1997-12-25  Quentin Tarantino
10                         Death Proof   2007-05-22  Quentin Tarantino
11    The Lost Chapter: Yuki's Revenge   2025-12-05  Quentin Tarantino
12                          Grindhouse   2007-04-06  Quentin Tarantino
13    

In [ ]:
print(search_by_actor("Leonardo DiCaprio"))

In [28]:
print(search_by_genre("Romance"))

                                    title release_date  \
0      Chainsaw Man - The Movie: Reze Arc   2025-09-19   
1                          Tell Me Softly   2025-12-11   
2                                 Dracula   2025-07-30   
3                        Wicked: For Good   2025-11-19   
4                Sidelined 2: Intercepted   2025-11-27   
...                                   ...          ...   
13653      Little Comedies in a Big House   1974-03-30   
13654   Hapax Legomena II: Poetic Justice   1972-02-16   
13655                  Hotarubi no Mori e   2011-09-17   
13656                            The Band   2009-11-17   
13657                               Score   1973-11-05   

                        director  
0              Tatsuya Yoshihara  
1      Denis Rovira van Boekholt  
2                     Luc Besson  
3                     Jon M. Chu  
4                      Justin Wu  
...                          ...  
13653        Aleksandr Shirvindt  
13654            Hollis

In [29]:
# now we have a recommendation system with search functionality based on director, actor, and genre
# we can now integrate this into our website using flask or django
# but first we save our models
import joblib
joblib.dump(nn_model, 'recommendation system.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

['tfidf_vectorizer.pkl']

In [1]:
print("leo " in "leonardo dicaprio")

False
